# SPX 0DTE throughput and latency diagnostic

## tl;dr

The evidence does **not** support relaxing the hard execution gates. In four health-complete sessions, the RTH funnel was **4 confirmed → 3 material intents → 2 trade-ready → 2 fresh exact L1 decisions → 1 strict replay fill**. The larger loss is upstream capture and execution-path latency, while GTH lacks forward exact-spread evidence entirely.

The service cadence improved from roughly 62 seconds on July 14–15 to roughly 5 seconds on July 16–17, but no new trade-ready event has yet validated the post-rollout end-to-end latency. The path is still a polled alert/virtual-execution loop rather than a broker execution lane. The safe capacity target is 5–10 qualified operations per week with a 2/day cap and valid no-trade days—not a forced daily quota.

## Context & Methods

Question: can the system reliably produce one or two high-quality SPX 0DTE operations per day, and are the current gates or latency the dominant constraint?

Method:

1. Freeze an exclusive cutoff at 2026-07-18 00:00 UTC.
2. Deduplicate detector output by event ID and service evaluations by event/intent ID.
3. Keep production trade-ready replay separate from the observational prefill proxy.
4. Measure RTH service cadence from the system journal at 13:30–20:00 UTC.
5. Treat exact-quote freshness, expiry, schema, coordinate and TTL as safety constraints; distinguish data unavailability from strategy rejection.

This notebook is diagnostic evidence, not a claim of strategy edge or investment advice. Automatic ordering remains disabled.

In [1]:
from collections import defaultdict
from csv import DictReader
from datetime import date, datetime, time, timezone
from pathlib import Path
from statistics import median
import glob
import json
import subprocess

from IPython.display import Markdown, display

DATA_ROOT = Path("/srv/data/spx-spark/data")
FEATURES = DATA_ROOT / "features"
BACKTEST_DIR = DATA_ROOT / "reports/odte_level_backtest/validation=2026-07-18-contract-v3/cutoff=2026-07-17"
CUTOFF_AT = datetime(2026, 7, 18, tzinfo=timezone.utc)
COMPLETE_SESSIONS = {"2026-07-14", "2026-07-15", "2026-07-16", "2026-07-17"}
RTH_START = time(13, 30, tzinfo=timezone.utc)
RTH_END = time(20, 0, tzinfo=timezone.utc)

for required in (
    FEATURES / "level_trigger_repricing",
    FEATURES / "trade_intents",
    FEATURES / "gth_dip_reclaim",
    FEATURES / "virtual_strategy",
    BACKTEST_DIR / "artifact.json",
    BACKTEST_DIR / "readiness.json",
    BACKTEST_DIR / "trades.csv",
):
    assert required.exists(), f"missing source: {required}"

print(f"Cutoff (exclusive): {CUTOFF_AT.isoformat()}")
print(f"Complete sessions: {', '.join(sorted(COMPLETE_SESSIONS))}")
print("Runtime evidence: systemd user journal for spx-spark-24h.service")

Cutoff (exclusive): 2026-07-18T00:00:00+00:00
Complete sessions: 2026-07-14, 2026-07-15, 2026-07-16, 2026-07-17
Runtime evidence: systemd user journal for spx-spark-24h.service


In [2]:
def parse_ts(value):
    if not value:
        return None
    parsed = datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        raise ValueError(f"naive timestamp: {value}")
    return parsed.astimezone(timezone.utc)


def load_jsonl(root, timestamp_fields):
    rows = []
    for path in sorted(root.glob("date=*/events.jsonl")):
        with path.open(encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, 1):
                row = json.loads(line)
                observed = next(
                    (parse_ts(row.get(field)) for field in timestamp_fields if row.get(field)),
                    None,
                )
                if observed is not None and observed >= CUTOFF_AT:
                    continue
                row["_source_path"] = str(path)
                row["_line_number"] = line_number
                row["_observed_at"] = observed
                rows.append(row)
    return rows


def markdown_table(headers, rows):
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    lines.extend("| " + " | ".join(str(value) for value in row) + " |" for row in rows)
    display(Markdown("\n".join(lines)))


def is_rth(value):
    observed = parse_ts(value)
    return observed is not None and RTH_START <= observed.timetz() < RTH_END


def quantile(values, probability):
    ordered = sorted(values)
    if not ordered:
        return None
    position = (len(ordered) - 1) * probability
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)

## Data

In [3]:
level_rows = load_jsonl(FEATURES / "level_trigger_repricing", ("as_of",))
intent_rows = load_jsonl(FEATURES / "trade_intents", ("evaluated_at",))
gth_rows = load_jsonl(FEATURES / "gth_dip_reclaim", ("confirmed_at",))
virtual_rows = load_jsonl(
    FEATURES / "virtual_strategy",
    ("opened_at", "observed_at", "closed_at", "at"),
)
with (BACKTEST_DIR / "artifact.json").open(encoding="utf-8") as handle:
    artifact = json.load(handle)
with (BACKTEST_DIR / "readiness.json").open(encoding="utf-8") as handle:
    readiness = json.load(handle)
with (BACKTEST_DIR / "trades.csv").open(newline="", encoding="utf-8") as handle:
    trades = list(DictReader(handle))

confirmed_rows = [row for row in level_rows if row.get("phase") == "confirmed"]
confirmed_by_event = defaultdict(list)
for row in confirmed_rows:
    confirmed_by_event[row["event_id"]].append(row)
confirmed_first = {
    event_id: min(rows, key=lambda row: row["as_of"])
    for event_id, rows in confirmed_by_event.items()
}

assert artifact["signal_counts"]["confirmed"] == len(confirmed_first) == 14
assert artifact["trade_intent_coverage"]["evaluation_records"] == len(intent_rows) == 474
assert artifact["window"]["complete_sessions"] == sorted(COMPLETE_SESSIONS)

markdown_table(
    ("source", "rows", "independent keys", "analytical role"),
    (
        ("level_trigger_repricing confirmed", len(confirmed_rows), len(confirmed_first), "candidate supply"),
        ("trade_intents", len(intent_rows), len({row.get('event_id') for row in intent_rows}), "decision telemetry"),
        ("GTH dip/reclaim", len(gth_rows), len({row.get('event_id') for row in gth_rows}), "legacy GTH signals"),
        ("virtual_strategy", len(virtual_rows), len({row.get('position_id') for row in virtual_rows if row.get('position_id')}), "virtual lifecycle"),
        ("strict replay trades", len(trades), len({row.get('key') for row in trades}), "replay ledger; mixed cohorts"),
    ),
)

| source | rows | independent keys | analytical role |
| --- | --- | --- | --- |
| level_trigger_repricing confirmed | 240 | 14 | candidate supply |
| trade_intents | 474 | 280 | decision telemetry |
| GTH dip/reclaim | 6 | 6 | legacy GTH signals |
| virtual_strategy | 32 | 0 | virtual lifecycle |
| strict replay trades | 259 | 19 | replay ledger; mixed cohorts |

Data-grain warning: detector lifecycle rows and repeated intent evaluations are telemetry, not independent trade opportunities. Partition dates also differ from payload session IDs in some rows, so joins use the payload identifiers and timestamps.

## Results

### RTH opportunity funnel

In [4]:
rth_event_ids = {
    event_id
    for event_id, row in confirmed_first.items()
    if is_rth(row["as_of"])
}
material_intents = [
    row
    for row in intent_rows
    if row.get("event_id") in rth_event_ids and row.get("status") != "observing"
]
material_event_ids = {row["event_id"] for row in material_intents}
ready_intents = [
    row
    for row in material_intents
    if row.get("status") == "trade_ready"
]
fresh_exact_l1 = []
for row in ready_intents:
    decision_at = parse_ts(row["evaluated_at"])
    quote_at = parse_ts(row.get("quote_source_at"))
    quote_age = (decision_at - quote_at).total_seconds() if quote_at else None
    if (
        row.get("provider")
        and row.get("contract_id")
        and row.get("decision_bid") is not None
        and row.get("decision_ask") is not None
        and quote_age is not None
        and -1.0 <= quote_age <= 5.0
    ):
        fresh_exact_l1.append(row)

strict_fills = artifact["production_strategy_total"]["result"]["n"]
funnel = [
    ("Confirmed RTH events", len(rth_event_ids)),
    ("Material intent persisted", len(material_event_ids)),
    ("Trade-ready", len({row['intent_id'] for row in ready_intents})),
    ("Fresh exact L1", len({row['intent_id'] for row in fresh_exact_l1})),
    ("Strict replay fill", strict_fills),
]
assert [value for _, value in funnel] == [4, 3, 2, 2, 1]

markdown_table(("stage", "independent opportunities", "conversion from RTH confirmed"), [
    (label, value, f"{value / funnel[0][1]:.0%}") for label, value in funnel
])
print("Observed strict fills per health-complete session:", strict_fills / len(COMPLETE_SESSIONS))
print("Observed trade-ready decisions per health-complete session:", len(ready_intents) / len(COMPLETE_SESSIONS))

| stage | independent opportunities | conversion from RTH confirmed |
| --- | --- | --- |
| Confirmed RTH events | 4 | 100% |
| Material intent persisted | 3 | 75% |
| Trade-ready | 2 | 50% |
| Fresh exact L1 | 2 | 50% |
| Strict replay fill | 1 | 25% |

Observed strict fills per health-complete session: 0.25
Observed trade-ready decisions per health-complete session: 0.5


The current strict result is 0.25 replay fills per complete session, versus the desired one or two operations per day. That gap is real, but the funnel loses one RTH event before any material intent exists; it is therefore not attributable solely to a strict strategy gate.

### Decision latency and quote freshness

In [5]:
latency_rows = []
for event_id in sorted(material_event_ids):
    confirmed_at = parse_ts(confirmed_first[event_id]["as_of"])
    terminal_candidates = [row for row in material_intents if row["event_id"] == event_id]
    final_row = max(terminal_candidates, key=lambda row: row["evaluated_at"])
    evaluated_at = parse_ts(final_row["evaluated_at"])
    quote_at = parse_ts(final_row.get("quote_source_at"))
    latency_rows.append((
        event_id,
        final_row["direction"],
        final_row["status"],
        f"{(evaluated_at - confirmed_at).total_seconds():.2f}",
        "—" if quote_at is None else f"{(evaluated_at - quote_at).total_seconds():.2f}",
        ", ".join(final_row.get("block_reasons", [])) or "—",
    ))

markdown_table(
    ("event", "direction", "final state", "confirmed→decision s", "quote age s", "final blocker"),
    latency_rows,
)

ready_delays = [float(row[3]) for row in latency_rows if row[2] == "trade_ready"]
assert [round(value, 2) for value in sorted(ready_delays)] == [47.45, 82.38]
assert sorted(round((parse_ts(row["evaluated_at"]) - parse_ts(row["quote_source_at"])).total_seconds(), 2) for row in ready_intents) == [0.73, 1.86]
print("Trade-ready confirmation delays:", sorted(round(value, 2) for value in ready_delays))
print("The final quotes were fresh; the end-to-end decision path was slow.")

| event | direction | final state | confirmed→decision s | quote age s | final blocker |
| --- | --- | --- | --- | --- | --- |
| level:0dbdd3ae2d08da30cf102b2d | down | blocked | 38.97 | — | es_return_1m_points_opposes_direction |
| level:84c4346b55f19eb00429c5a6 | down | trade_ready | 47.45 | 0.73 | — |
| level:a2799ffe7e2fe7a8516e59e5 | up | trade_ready | 82.38 | 1.86 | — |

Trade-ready confirmation delays: [47.45, 82.38]
The final quotes were fresh; the end-to-end decision path was slow.


### Runtime cadence by RTH day

In [6]:
journal_text = subprocess.check_output(
    [
        "journalctl", "--user", "-u", "spx-spark-24h.service",
        "--since", "2026-07-14 00:00:00 UTC",
        "--until", "2026-07-18 00:00:00 UTC",
        "--no-pager", "-o", "cat",
    ],
    text=True,
)
journal_rows = []
for line in journal_text.splitlines():
    try:
        row = json.loads(line)
    except json.JSONDecodeError:
        continue
    if row.get("task") != "market_features" or "duration_ms" not in row:
        continue
    finished_at = parse_ts(row["finished_at"])
    if RTH_START <= finished_at.timetz() < RTH_END:
        journal_rows.append((finished_at, float(row["duration_ms"]) / 1000.0))

cadence_rows = []
for session_day in range(14, 18):
    samples = sorted(row for row in journal_rows if row[0].day == session_day)
    gaps = [(current[0] - previous[0]).total_seconds() for previous, current in zip(samples, samples[1:])]
    durations = [row[1] for row in samples]
    cadence_rows.append((
        f"2026-07-{session_day:02d}",
        len(samples),
        round(median(gaps), 3),
        round(quantile(gaps, 0.95), 3),
        round(quantile(gaps, 0.99), 3),
        round(median(durations), 3),
        round(quantile(durations, 0.95), 3),
        sum(gap > 10.0 for gap in gaps),
        round(max(gaps), 3),
    ))

assert [round(row[2], 2) for row in cadence_rows] == [62.09, 62.74, 5.01, 5.01]
markdown_table(
    ("RTH session", "runs", "start gap p50 s", "start gap p95 s", "start gap p99 s", "duration p50 s", "duration p95 s", "gaps >10 s", "max gap s"),
    cadence_rows,
)

| RTH session | runs | start gap p50 s | start gap p95 s | start gap p99 s | duration p50 s | duration p95 s | gaps >10 s | max gap s |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-07-14 | 378 | 62.089 | 63.058 | 63.638 | 1.984 | 2.787 | 377 | 84.133 |
| 2026-07-15 | 371 | 62.744 | 64.125 | 65.214 | 2.455 | 3.657 | 370 | 79.871 |
| 2026-07-16 | 4471 | 5.006 | 6.36 | 10.091 | 2.735 | 4.654 | 47 | 106.629 |
| 2026-07-17 | 4551 | 5.009 | 6.104 | 7.219 | 3.262 | 4.885 | 2 | 13.015 |

The July 16 rollout materially improved polling/sampling cadence; it does not by itself prove better opportunity detection. The new five-second cadence has no trade-ready sample for end-to-end validation, and it did not create a real-time execution lane: the loop still polls, shares workers with slower tasks, and performs decision-adjacent work before virtual execution. The journal field named `finished_at` is currently emitted before execution, so the gaps above are scheduler start-to-start gaps; `duration_ms` remains task duration. The actionable SLO must measure detector → intent → quote → persisted action, not only quote age.

### GTH exact-execution readiness

In [7]:
gth_signal_ids = {row["event_id"] for row in gth_rows}
gth_opens = [
    row for row in virtual_rows
    if row.get("event") == "virtual_opened" and row.get("source_kind") == "gth_dip_reclaim_call"
]
gth_open_ids = {row.get("source_signal_id") for row in gth_opens}

assert len(gth_signal_ids) == 6
assert gth_open_ids == gth_signal_ids
assert readiness["cohorts"]["gth_exact_entry"]["count"] == 0

markdown_table(
    ("measure", "observed", "interpretation"),
    (
        ("Legacy GTH signals", len(gth_signal_ids), "candidate supply exists"),
        ("Legacy virtual opens", len(gth_open_ids), "not exact two-leg execution evidence"),
        ("Forward v3 exact GTH entries", readiness["cohorts"]["gth_exact_entry"]["count"], "cannot assess gate economics"),
        ("Forward contract-consistent sessions", readiness["sessions"]["contract_consistent_complete"], "20-session review has not started accumulating"),
    ),
)

| measure | observed | interpretation |
| --- | --- | --- |
| Legacy GTH signals | 6 | candidate supply exists |
| Legacy virtual opens | 6 | not exact two-leg execution evidence |
| Forward v3 exact GTH entries | 0 | cannot assess gate economics |
| Forward contract-consistent sessions | 0 | 20-session review has not started accumulating |

The GTH 5-second quote gate should not be widened to make stale data pass. Historical IBKR rotation and Schwab GTH snapshots do not reliably supply both exact legs inside five seconds. The correct engineering response is to pin and pre-warm the intended long/short legs while the dip/reclaim signal is pending, then shadow-test 5/10/15-second counterfactual coverage.

### Gate disposition and operating target

In [8]:
markdown_table(
    ("class", "examples", "runtime treatment", "decision"),
    (
        ("Safety", "schema/policy/coordinate/expiry/TTL; exact NBBO; max loss", "blocked_safety", "keep hard"),
        ("Data acquisition", "structure/candidate/quote unavailable", "pending_data + retry within TTL", "do not count as strategy rejection"),
        ("Strategy", "1m ES opposition; redundant micro-confirmations", "rejected_strategy + shadow counterfactual", "change only after forward evidence"),
        ("Research readiness", "20 complete sessions / exact-spread cohorts", "offline promotion gate", "must not suppress runtime signals"),
    ),
)

markdown_table(
    ("operating metric", "target", "current evidence"),
    (
        ("Qualified operations", "5–10/week; cap 2/day; no-trade day valid", "1 strict replay fill / 4 complete sessions"),
        ("Valid-signal capture", ">=90%", "3/4 RTH confirmed reached a material intent"),
        ("Detector→intent", "p95 <1s excluding explicit hold", "47.45s and 82.38s for two ready events"),
        ("Fresh quote→persist", "p95 <0.5s", "not instrumented end-to-end"),
        ("RTH service cadence", "p95 gap <=5.5s; none >10s", "Jul-17 p95 6.10s; 2 gaps >10s; max 13.02s"),
        ("GTH exact-spread evidence", ">=20 entries before economic promotion", "0"),
    ),
)

| class | examples | runtime treatment | decision |
| --- | --- | --- | --- |
| Safety | schema/policy/coordinate/expiry/TTL; exact NBBO; max loss | blocked_safety | keep hard |
| Data acquisition | structure/candidate/quote unavailable | pending_data + retry within TTL | do not count as strategy rejection |
| Strategy | 1m ES opposition; redundant micro-confirmations | rejected_strategy + shadow counterfactual | change only after forward evidence |
| Research readiness | 20 complete sessions / exact-spread cohorts | offline promotion gate | must not suppress runtime signals |

| operating metric | target | current evidence |
| --- | --- | --- |
| Qualified operations | 5–10/week; cap 2/day; no-trade day valid | 1 strict replay fill / 4 complete sessions |
| Valid-signal capture | >=90% | 3/4 RTH confirmed reached a material intent |
| Detector→intent | p95 <1s excluding explicit hold | 47.45s and 82.38s for two ready events |
| Fresh quote→persist | p95 <0.5s | not instrumented end-to-end |
| RTH service cadence | p95 gap <=5.5s; none >10s | Jul-17 p95 6.10s; 2 gaps >10s; max 13.02s |
| GTH exact-spread evidence | >=20 entries before economic promotion | 0 |

## Takeaways

1. **The user's latency concern is correct.** Historical cadence was around 62 seconds and the two ready decisions arrived 47–82 seconds after confirmation. The current five-second loop is better but has no new trade-ready sample for end-to-end validation and is still not an event-driven execution path.
2. **Hard gates are not the demonstrated throughput bottleneck.** One of four RTH confirmations never produced any material intent, while GTH has zero forward exact-spread entries. These are capture/data-path failures.
3. **Do not force one or two trades every day.** Use a weekly capacity target of 5–10 qualified operations with a two-per-day cap; allow zero when no setup survives safety and quality checks.
4. **P0 engineering:** persist the deterministic decision first, move LLM/notifications to an async outbox, use a fresh wall clock at the action point, isolate the hot worker, and pre-subscribe exact GTH legs.
5. **Only soft gates should enter shadow relaxation.** Record counterfactual outcomes for the ES 1-minute opposition rule and redundant confirmation checks, then decide after at least 20 forward complete sessions and exact-spread evidence.

Confidence is high for event counts and measured latency, and low for any claim that relaxing a strategy rule would improve expectancy because the forward v3 economic sample is zero.